In [3]:
import pandas as pd
from pathlib import Path
import numpy as np
import json

In [4]:
OUTPUT_PATH = Path("dati_schede_min.csv")
df = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False)
df.head()

,id_scheda_originale,numero_catalogo_iccd,categoria_edificio,tipologia_edificio,numero_spazi,regione,sigla_provincia,comune,contesto_paesaggistico,tipo_dissesto_idrogeologico,...,anno_compilazione_scheda,nome_compilatore,ente_compilatore,nome_responsabile_scientifico,ruolo_responsabile_scientifico,ente_responsabile_scientifico,profilo_pubblicazione,titolo,descrizione,gaussian
0,ce9c9265-54dd-4779-9c4b-cc282dc908e3,182407,EDIFICIO SINGOLO,casa con loggiato,nr,Friuli-Venezia Giulia,UD,Dignano,pianura,alluvione,...,2024.0,"Rulli, Eduardo",Atlante Group SPA,"Parisi, Valeria",responsabile coordinamento delle attività,Atlante Group s.r.l.,2.0,,"A un’attenta osservazione del prospetto sud, l...",0
1,6e02ff7c-2b9d-40d0-a14c-e8f4d03d9a83,252733,EDIFICIO SINGOLO,casa a corte chiusa,nr,Sardegna,SU,Soleminis,collina,nessun dissesto evidente,...,2025.0,"Camatti, Matteo",Progetto PSC,"cesarano, barbara",responsabile coordinamento delle attività,progetto psc,1.0,,Casa a corte retrostante ad uso abitativo (1a-...,0
2,814d64dd-d0f5-4c01-ace8-968ec355547e,223849,EDIFICIO CON ANNESSI,casa a scala esterna,1,Basilicata,PZ,San Martino d'Agri,altopiano,nessun dissesto evidente,...,2024.0,"Trausi, Pier Pasquale",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle attività,GLOSSA Srl,2.0,Masseria Rubalo,"Casa di abitazione rurale di tipo ""collinare"" ...",0
3,7aea73d3-286c-4fa1-aade-0883bd9f9c78,182399,EDIFICIO CON ANNESSI,cascina,2,Friuli-Venezia Giulia,UD,Comune di Pavia di Udine,pianura,alluvione,...,2024.0,"Rulli, Eduardo",Atlante Group SPA,"Parisi, Valeria",responsabile coordinamento delle attività,Atlante Group s.r.l.,2.0,,"Oltre al corpo di fabbrica principale, a compl...",0
4,d8d256d7-e480-40c6-bd8a-0d8d4dd66f45,310548,COMPLESSO,casa a edifici affiancati,2,Abruzzo,TE,Castelli,collina,nessun dissesto evidente,...,2025.0,"Frezzini, Luca",Unimol,"Alessandria, Francesco",responsabile coordinamento delle attività,Cles,2.0,Colledoro,Edificio a pianta rettangolare orientato in di...,0


In [5]:
#count rows and where gaussian column is 1

print("number of rows: {}".format(len(df)), "number of data with gaussian splat: {}".format(len(df[df['gaussian'] == '1'])))


number of rows: 44363 number of data with gaussian splat: 403


In [6]:
df["stato_conservazione"].value_counts()

stato_conservazione
discreto                14869
buono                   11337
mediocre                10023
cattivo                  4504
pessimo                  2510
                         1100
dato non disponibile       20
Name: count, dtype: int64

In [7]:
#print the last 50 titles from the schede that have buono or discreto stato_conservazione AND a non empty titolo
df_good_condition = df[df["stato_conservazione"].isin(["buono", "discreto"]) & (df["titolo"].str.strip() != "")]
print(f"Number of schede with buono or discreto stato_conservazione and non empty titolo: {len(df_good_condition):,}")
print("\nLast 50 titles:")
for i, titolo in enumerate(df_good_condition["titolo"].tail(50), start=1):
    print(f"{titolo}") 

Number of schede with buono or discreto stato_conservazione and non empty titolo: 5,598

Last 50 titles:
La Concia
Masseria Montanaro
Case dell'Oliveto
Casale del Fornaccio
Casino Dattilo
Case Pozzello
Casale della Vannina
C.Le Pagliarini
Cascina Sorianino
ex Ost.a dell'Ellera
Masseria Magistro
Casa Agrillusa
Casa Contrada Pratelli
Casale della Volpe
Case Morolo
C. Facchinaccia
C.Le Testaccio
Cascina Cantalupo
Villa Achille Albanese
corte Preatoni
Casa Ottaviani
Casa Cepa
Casone Bianco
Cascina Ulivieri
Casa Ferrara
Villa Spina
Cascina Pedaggera
Casa Firriato-Imbornone-Napolitani
Casa Piacentino
Casa Rizzo-Muegen
Villa Trabia
Tenuta Borgia
Case Scuderi
Baglio Ingardia Fontansalsa
Palazzo Viani Tagliavacca
Villa Serraino
Baglio Fontanasalsa
Cascina Ghiringhella
Podere Casa Grande
Casa Ingardia Misiliscemi
Baglio in Via Federico II Stupor Mundi -Nubia
Il baglio di Pantelleria
Cascina Scanna
Baglio Sanacore
Villa Funtanazzi
Casa Messina
Villa Raffo
Casa Gugliatore
Casa S. Iorio
Villa Jacon

In [8]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

LANDSCAPE_MAP = {
    "costa (bassa)": "costa",
    "costa (alta, falesie)": "costa",
    "valle": "valle e fondovalle",
    "fondovalle": "valle e fondovalle",
    "conca intramontana": "valle e fondovalle",
    "pedemontano": "valle e fondovalle",
    "lagunare": "zone umide e acque interne",
    "lacustre": "zone umide e acque interne",
    "palustre": "zone umide e acque interne",
    "foce fluviale": "zone umide e acque interne",
    "montagna": "montagna",
    "versante ripido": "montagna",
    "crinale/dorsale": "montagna",
    "versante a debole pendenza": "collina",
    "collina": "collina",
    "carsico (doline, cavità ipogee)": "carsico",
    "altopiano": "altopiano",
    "pianura": "pianura",
}

raw_counts = df["contesto_paesaggistico"].value_counts()

df["contesto_paesaggistico_ridotto"] = df["contesto_paesaggistico"].map(LANDSCAPE_MAP)

reduced_counts = df["contesto_paesaggistico_ridotto"].value_counts()
print("\n=== contesto_paesaggistico_ridotto (x8) ===")
print(reduced_counts.to_string())




=== contesto_paesaggistico_ridotto (x8) ===
contesto_paesaggistico_ridotto
collina                       16961
pianura                       14236
montagna                       5412
valle e fondovalle             4972
altopiano                       917
costa                           400
zone umide e acque interne      230
carsico                         134


In [9]:
# give me a list of first 10 id_schede that have contesto paesaggistico = "altopiano" and gaussian as 1
df[(df["contesto_paesaggistico"] == "altopiano") & (df["gaussian"] == "1")]["id_scheda_originale"].head(10)

1395     dbb213a4-c091-4685-bb2d-27d192028206
6143     d2ab0fd0-84e7-402f-9a31-2a1c03155be0
8901     ac2e862f-7ccf-49ce-9c2b-a849c94d8674
19748    4b7752de-622e-4116-ab78-813ef608958b
24259    30a1ac31-b90c-4bb7-a875-ab3b12f82f09
28385    309ce237-14f3-4843-9935-3d6bf014b530
42468    15878fc6-895c-46e1-9c44-eebdf82ac276
42538    036eb456-54ba-4155-9fec-99a26d9a8aaf
42989    bd5cc795-5465-43a3-b491-04adeb030a6d
43847    d6b9a4ee-0565-4c63-8a98-c5c18549081e
Name: id_scheda_originale, dtype: str

In [10]:
# good cool houses
belle = {
    "0a88637e-305b-4658-af0f-cb04a4db9cbf", "0dea2c1c-3bb4-4586-a329-c73a4fcf5dd8",
    "0ec34e16-35c5-4910-9738-a2662f241b2c", "0ee51f3c-23f1-49fe-9147-bd1b61b0b0ac",
    "1b415eb8-6dad-441f-89d9-52218ec8a75d", "1be76e2a-a2a2-46aa-9017-2a3911eae468",
    "1e55b249-20af-48ba-bf26-06fd9c4358f8", "2ab5d014-5d5a-4df9-baa3-9c8a861a0053",
    "2b0f3bae-705b-460b-9f44-11a84848197a", "2d4c709f-71b6-4746-9507-61e6342105c3",
    "2db5e657-6e25-442f-9cc4-937a447f3e23", "2e3dc4c3-7466-4ca6-9784-b8e5ba8a7c25",
    "2f9e2a1d-9cab-4c19-8ade-e48d4979b501", "2faeb311-d93a-402a-8edb-0fc5f231cb36",
    "03d8a957-5cc3-4da8-a160-2b733143771a", "3ad076d5-1819-4cb6-a0dc-0b80337dca35",
    "3c51f577-1d38-4f82-b6cf-50402bc4dfe3", "3cd4dc4b-0add-4e0c-8ddd-3c460342a2c6",
    "3cedb99a-af79-45bf-a427-c0df70673b29", "3e2be11e-aa89-43c5-b790-b1df4abccc14",
    "3e6cc735-51b1-46aa-825e-56bbaa69f5f0", "3e1330d2-1f11-4a38-b1b1-f9f87e227fd9",
    "3ef1dc21-f9fa-4f24-b02b-5ca7696ba476", "3f05f86a-4a1e-48ee-b884-0bcaff4512c2",
    "4ae9b11b-3422-41e1-bae9-243df2d32ff1", "4c0ce6f0-acbd-4581-93a8-5b2a27ce77f9",
    "4c61e522-4905-4e02-886f-a2eca48d3de4", "4d188656-e812-45f4-9dbb-ee6e84ae578e",
    "4fb624f3-0a69-46e9-9003-65063862e780", "05d3a360-1671-43f6-8beb-4babef434277",
    "5a0ec71b-4b44-4364-a1c0-911b83e3e7b7", "5a89477f-5a10-4ccd-a196-7fa7fbdbed55",
    "5b28f6a0-2693-4d86-a73f-2b3b50a1ca96", "5da748f9-addb-430d-aa15-31a86d27749c",
    "5e0ac060-f3bb-4b92-88d1-a90426edcf57", "06c87500-7a78-45be-8fe0-90ba3b310535",
    "6a2f61f8-2c67-4c73-91ba-f83b6912486d", "6ad295a2-8456-478d-8919-5f3f134d8a77",
    "6b8482d8-3ab9-42e2-8051-106d3514abe6", "6ba97814-fc00-4e33-b338-2a548da306e8",
    "6c0505a7-6afa-42cf-af69-a568acca04bc", "6dc2713a-2bdf-4565-977b-620fbf916751",
    "6ddde6ea-193d-4860-931e-9b396d590c23", "6dea1a8c-be95-4c9c-ae3c-d3a8239c051b",
    "6dedc206-5501-43ef-b3ff-421137532431", "7a27c0cc-1e2a-4016-b48d-dadf0b736839",
    "7e3cd0f5-8b72-41aa-bce9-6a6e62fb679c", "8b525e04-6ad8-4447-9c24-4c5c52595fae",
    "8b5147ba-f00d-47ca-8b2c-84b64febe89e", "8ee5e18b-4f86-4833-82f8-736b02d84ad4",
    "8f7d7a47-9176-4f48-af86-7721b893b19b", "09c65d5b-8a95-49d5-a645-ede008a2c152",
    "09d75789-bea3-4b54-9bbb-9358510e27a1", "9a47df01-c4d6-4117-81f8-bb30a1ddd8a4",
    "9b849c4c-18b3-468e-a70f-dc904332a708", "9b8460ac-1a5f-4e81-a568-bd8607aca73b",
    "9c127a30-deb0-4dc0-b366-774a0d6f2bc7", "9e658dbb-cb23-4d6a-861e-05f5871be7b4",
    "14a7780b-f008-4007-85ec-3bf1ff90efa3", "14ff4360-aa20-4d2a-921b-b0f86e119771",
    "15d021d3-dd0d-46bc-948b-ed1c9ddce6a3", "018a4cb2-a88f-4740-92f0-7cc61a152e51",
    "22d8e178-2500-4e54-ae8d-225b11392aca", "28e139a0-20b2-4cff-a322-eff45fb2892f",
    "30a1ac31-b90c-4bb7-a875-ab3b12f82f09", "48b6f691-91e0-4836-9b08-ce45b780864a",
    "52e80d0a-c2be-4179-bc3f-195c66346b80", "55c5a507-edae-4a4f-8083-eb192b22fa49",
    "67f393eb-75ec-42fd-b24d-24affb8d2056", "71a40bc3-5fcb-4099-8cf2-14c7e3bad777",
    "72f7d0e5-01f1-4241-aec3-40fe838728ff", "75b27192-4b7e-41ba-b837-cbed796604e2",
    "79a617d8-85a3-42d6-9c31-eda77ab525b2", "82daa58a-56f6-42e7-bf25-1e7b560717fb",
    "83b268bc-a75c-4a68-b05b-0516681afe30", "93cf5bc9-b93f-4299-a97e-14ee01ad8387",
    "96ace98f-9aa4-4ab3-b137-114ffc2d75a9", "97eed373-723b-4be5-85f8-727f1d133839",
    "144db232-5705-4824-a325-d38b8f51875c", "309ce237-14f3-4843-9935-3d6bf014b530",
    "380e406c-c95a-4100-89af-1da30832913a", "554bb976-4488-4064-b6fa-b5f91c1063d9",
    "559c06ea-79a7-4769-8933-d5da50668dea", "863fca31-1150-42c0-bd79-6927f5eca859",
    "951dc9bc-7715-4b0c-8020-144e405a24f6", "3353da4e-1c65-4dbc-b189-1b6cb6f3383e",
    "7294b34d-752f-4bad-8806-118469997aa3", "11116fd9-3270-4110-8fee-37b705e6389b",
    "49304e13-ef97-4db4-8f85-0affb7290c79", "60511cf4-8309-4244-91d9-a1e0c0cafcdd",
    "070189cb-0f0a-4eb1-a17b-6e35e8c97176", "91175aaa-b748-40f7-9be3-85f9e1dd47fb",
    "139918d7-9a02-49cf-994e-f7f42a23e2e4", "216708f3-e659-4373-9b29-5a4fbaa9b002",
    "414126fe-6fc6-471e-8ca8-23291ac54764", "426052aa-4f02-431f-b507-dbe38b0e9137",
    "482924b8-90a4-4bd2-a8b0-71b9319178ce", "677469f2-bf4d-4cfc-9163-f9ce5065d99a",
    "a0e0abe4-e37e-4b12-b5c1-f504ecb8951e", "a6d7393d-e059-4512-ae62-704de8090d43",
    "a8bdca43-3ddb-418f-ac34-42df35b9dce0", "a9aba613-bd87-4238-83a5-e617cc6a710b",
    "a647f319-24a6-47e5-8f75-2ced8a76c165", "ac2e862f-7ccf-49ce-9c2b-a849c94d8674",
    "b3dc0422-212d-4daf-97ea-77f462e5ce23", "b5b0e8bc-a1a7-4496-8ce8-cb2d3fca5daf",
    "b28985ec-48c0-42d5-b474-96b797ee78c3", "c6fc1c30-a88a-4dff-98e1-c4c46c97828b",
    "c7ebfc27-ac20-4612-8fde-1e7f1604c501", "c387257e-fa24-42e5-a800-3c0608d994ee",
    "cb817635-3e4f-4e0d-9f7d-f89ee23fce30", "d6b9a4ee-0565-4c63-8a98-c5c18549081e",
    "d4811010-b592-485f-ac86-2768a62eaf2f", "db429e09-a2b4-48ed-9da6-3a59c6cccb81",
    "e1f4f488-64e5-4358-9446-e04cc79da6fd", "e5abb7b9-c8f2-4e75-b63c-b78bb40d54a1",
    "e44c03ea-ea29-4207-8e0b-be7fd9b7185f", "e52e1d5f-5b75-4b19-9662-edb313877d10",
    "e77da5fa-82aa-4e48-81f1-3b0fb8f4cdd7", "e7456750-ce6f-45ad-b9ff-6a1521361bc7",
    "f69efa2e-f3a5-4035-8ebb-6a23d358c42b", "f675b6c7-78aa-4d8a-a2cc-c32232cd418d",
    "f083655b-b103-4b6a-9710-026afbfec499", "fbbd9c47-12f0-4d02-91b1-84d7659caffe",
    "fd18ccb7-5dc5-4b77-b303-f0b2f9caa292", "fe9285e7-cbcd-48be-82bc-86bd4b5d17cb",
    "ff8e81f1-53a5-46cb-8c4a-b10d96a6428c", "ff450bbe-79a7-46ab-b105-4a4ece01a3de",
}

In [20]:
# --- Filter dataset by gaussian_ids ---

# sanity check on column count
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

# filter rows whose id_scheda_originale is in gaussian_ids
df_good = df[df["id_scheda_originale"].isin(belle)].copy()

print(f"Matched {df_good.shape[0]} / {len(belle)} ids")

# check for any ids in the set that were NOT found in the csv
missing_ids = belle - set(df_good["id_scheda_originale"])
if missing_ids:
    print(f"Warning: {len(missing_ids)} ids not found in csv:")
    for mid in missing_ids:
        print(" -", mid)

df_good.reset_index(drop=True, inplace=True)

# truncate long column values for display purposes only
with pd.option_context("display.max_colwidth", 40):
    display(df_good.head(5))

Loaded 44363 rows, 30 columns
Matched 128 / 128 ids


,id_scheda_originale,numero_catalogo_iccd,categoria_edificio,tipologia_edificio,numero_spazi,regione,sigla_provincia,comune,contesto_paesaggistico,tipo_dissesto_idrogeologico,...,nome_compilatore,ente_compilatore,nome_responsabile_scientifico,ruolo_responsabile_scientifico,ente_responsabile_scientifico,profilo_pubblicazione,titolo,descrizione,gaussian,contesto_paesaggistico_ridotto
0,9a47df01-c4d6-4117-81f8-bb30a1ddd8a4,224770,COMPLESSO,masseria,2,Basilicata,PZ,Marsico Nuovo,montagna,nessun dissesto evidente,...,"Cerone, Costabile",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle att...,GLOSSA Srl,1.0,Masseria Russo,L’edificio a pianta rettangolare con...,1,montagna
1,83b268bc-a75c-4a68-b05b-0516681afe30,252735,EDIFICIO CON ANNESSI,casa a corte chiusa,2,Sardegna,NU,Atzara,collina,nessun dissesto evidente,...,"Camatti, Matteo",Progetto PSC,"cesarano, barbara",responsabile coordinamento delle att...,progetto psc,1.0,,"L'annesso, ad uso deposito, è costit...",1,collina
2,863fca31-1150-42c0-bd79-6927f5eca859,619120,EDIFICIO CON ANNESSI,maso,1,Trentino-Alto Adige,BZ,Gais,valle,frana,...,"Fortunato, Matilde",Atlante Group s.r.l.,"Parisi, Valeria",responsabile coordinamento delle att...,Atlante Group s.r.l.,2.0,,L’edificio si trova in provincia di ...,1,valle e fondovalle
3,2e3dc4c3-7466-4ca6-9784-b8e5ba8a7c25,1412498,EDIFICIO SINGOLO,casale,nr,Lazio,RM,Pomezia,valle,nessun dissesto evidente,...,"Vaccariello, Alessia",GLOSSA Srl,"Mangone, Fabio",responsabile verifica scientifica,GLOSSA Srl,2.0,,L'edificio si struttura in due porzi...,1,valle e fondovalle
4,55c5a507-edae-4a4f-8083-eb192b22fa49,223823,EDIFICIO SINGOLO,casa a scala esterna,nr,Basilicata,PZ,Roccanova,collina,nessun dissesto evidente,...,"D'Angiulli, Giuseppe",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle att...,GLOSSA Srl,2.0,Masseria Picone,"Masseria rurale unitaria di tipo ""co...",1,collina


In [17]:
# get rows in df_good that have contesto_paesaggistico = "altopiano" and titolo non empty
df_altopiano = df_good[(df_good["contesto_paesaggistico"] == "altopiano") & (df_good["titolo"].str.strip() != "")]
df_altopiano.reset_index(drop=True, inplace=True)

# truncate long column values for display purposes only
with pd.option_context("display.max_colwidth", 40):
    display(df_altopiano.head(5))


,id_scheda_originale,numero_catalogo_iccd,categoria_edificio,tipologia_edificio,numero_spazi,regione,sigla_provincia,comune,contesto_paesaggistico,tipo_dissesto_idrogeologico,...,nome_compilatore,ente_compilatore,nome_responsabile_scientifico,ruolo_responsabile_scientifico,ente_responsabile_scientifico,profilo_pubblicazione,titolo,descrizione,gaussian,contesto_paesaggistico_ridotto
0,30a1ac31-b90c-4bb7-a875-ab3b12f82f09,402672,COMPLESSO,masseria,5,Puglia,BA,Conversano,altopiano,nessun dissesto evidente,...,"Iudice, Caterina Alma",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle att...,GLOSSA Srl,2.0,Masseria Franchini,"La stalla, di ampie dimensioni e att...",1,altopiano
1,309ce237-14f3-4843-9935-3d6bf014b530,108853,EDIFICIO SINGOLO,casa a scala esterna,nr,Molise,CB,Oratino,altopiano,nessun dissesto evidente,...,"Venditto, Roberta",Unimol,"Alessandria, Francesco",responsabile coordinamento delle att...,Cles,1.0,Casa Pozzo Nuovo,Dimora monofamiliare in pietra local...,1,altopiano


In [19]:
# get rows in df_good that have contesto_paesaggistico = "collina" and titolo non empty
df_collina = df_good[(df_good["contesto_paesaggistico"] == "collina") & (df_good["titolo"].str.strip() != "")]
df_collina.reset_index(drop=True, inplace=True)

# truncate long column values for display purposes only
with pd.option_context("display.max_colwidth", 40):
    display(df_collina.head(5))


,id_scheda_originale,numero_catalogo_iccd,categoria_edificio,tipologia_edificio,numero_spazi,regione,sigla_provincia,comune,contesto_paesaggistico,tipo_dissesto_idrogeologico,...,nome_compilatore,ente_compilatore,nome_responsabile_scientifico,ruolo_responsabile_scientifico,ente_responsabile_scientifico,profilo_pubblicazione,titolo,descrizione,gaussian,contesto_paesaggistico_ridotto
0,55c5a507-edae-4a4f-8083-eb192b22fa49,223823,EDIFICIO SINGOLO,casa a scala esterna,nr,Basilicata,PZ,Roccanova,collina,nessun dissesto evidente,...,"D'Angiulli, Giuseppe",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle att...,GLOSSA Srl,2.0,Masseria Picone,"Masseria rurale unitaria di tipo ""co...",1,collina
1,60511cf4-8309-4244-91d9-a1e0c0cafcdd,223927,COMPLESSO,masseria,5,Basilicata,MT,Comune di Matera,collina,nessun dissesto evidente,...,"Trausi, Pier Pasquale",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle att...,GLOSSA Srl,2.0,Masseria Castiglione,"La struttura annessa, adiacente all'...",1,collina
2,06c87500-7a78-45be-8fe0-90ba3b310535,108871,EDIFICIO CON ANNESSI,casale,2,Molise,CB,Guardiaregia,collina,nessun dissesto evidente,...,"Sabatini, Francesca",Unimol,"Alessandria, Francesco",responsabile coordinamento delle att...,Cles,1.0,Casa Giardini Zaffiro,Il secondo corpo di fabbrica (2 nell...,1,collina
3,0ee51f3c-23f1-49fe-9147-bd1b61b0b0ac,178863,COMPLESSO,masseria a corte,1,Calabria,KR,Cutro,collina,nessun dissesto evidente,...,"Carricola, Caterina",Cooperativa Archeologia-Komedia-Cori...,"Porcile, Marta",responsabile coordinamento delle att...,Cooperativa Archeologia-Komedia-Cori...,1.0,Casino di Doria,Casino a corte chiusa composto da re...,1,collina
4,3f05f86a-4a1e-48ee-b884-0bcaff4512c2,228661,EDIFICIO CON ANNESSI,casa,4,Umbria,TR,Comune di Avigliano Umbro,collina,nessun dissesto evidente,...,"Ratini, Filippo",Università degli Studi di Roma Tor V...,"Alessandria, Francesco",responsabile coordinamento delle att...,Cles,2.0,C. Piana I,Manufatto posto a poca distanza dall...,1,collina


In [21]:
!pip freeze > requirements.txt